# Imports

In [ ]:
import pathlib
import pandas as pd
import numpy as np

from experiments.models import (
    AutoETSForecast
)

from experiments.utils import load_experiments_specs

# Load Experiment Specifications

In [ ]:
dataset = "ABC"

In [ ]:
# Load specifications
train_type = "global"
experiment_specs = load_experiments_specs(
    dataset=dataset,
    train_type=train_type,
)

# Train and Test Data
train_df = experiment_specs["train"]
test_df = experiment_specs["test"]

# Meta Data
meta = experiment_specs["meta"]
fcst_h = meta["fcst_h"]
freq = meta["freq"]
max_lag = meta["max_lag"]
seed = 123

# ETS Model

In [ ]:
np.random.seed(seed)

# Auto-ETS
auto_ets_fcst = AutoETSForecast(
    train_df,
    test_df.drop(columns=["value"]),
    fcst_h,
    freq,
    season_length=max_lag
)

# Combine Forecasts

In [ ]:
# Add actual values and dataset information
fcsts_df = pd.merge(
    auto_ets_fcst,
    test_df[["dataset", "series_id", "date", "value"]],
    on=["series_id", "date"],
    how="inner"
)

# Save Forecasts

In [ ]:
repo_root = pathlib.Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

result_path = repo_root / "experiments" / "runs" / "results" / train_type
result_path.mkdir(parents=True, exist_ok=True)

fcsts_df.to_csv(result_path / f"{dataset.lower()}_ets_fcsts.csv", index=False)